# Used Cars Cost Prediction

**Acmegrade internship project â€” beginner walkthrough**

This notebook uses the real public CarDekho dataset to estimate `Selling_Price`. It walks through loading the data, checking and cleaning it, plotting exploratory charts, fitting a regression model, evaluating it on held-out data, and summarizing the results. No sample or synthetic data is used. If the CSV is missing, the loading cell stops and points you to `data/README.md`.

**Dataset expected:** `data/car data.csv` with a column named exactly `Selling_Price`.

## 1. Import the tools

Pandas works with tables, Matplotlib and Seaborn make charts, and scikit-learn provides the train/test split, preprocessing, model, and evaluation measures. Run this notebook using the project virtual environment described in the project README.

In [ ]:
from pathlib import Path
import sys

try:
    import joblib
    import matplotlib.pyplot as plt
    import pandas as pd
    import seaborn as sns
    from sklearn.compose import ColumnTransformer
    from sklearn.ensemble import RandomForestRegressor
    from sklearn.impute import SimpleImputer
    from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
    from sklearn.model_selection import train_test_split
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import OneHotEncoder
except ImportError as exc:
    raise ImportError(
        f"Missing package: {exc.name}. Activate the project environment and "
        "install packages with `python -m pip install -r requirements.txt`."
    ) from exc

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent
DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "submission"
TARGET = "Selling_Price"
sns.set_theme(style="whitegrid")
print(f"Project folder: {PROJECT_DIR}")

## 2. Load the real dataset

The notebook looks for CSV files in `data/` and selects one containing the expected `Selling_Price` target. If it cannot find one, download the dataset using the instructions in `data/README.md`; do not replace the missing dataset with made-up rows.

In [ ]:
csv_files = sorted(DATA_DIR.glob("*.csv"))
if not csv_files:
    raise FileNotFoundError(
        f"No CSV file found in {DATA_DIR}. Obtain the real CarDekho CSV as explained "
        "in data/README.md, put it in the data folder, then rerun this cell."
    )

cars = None
for csv_path in csv_files:
    try:
        candidate = pd.read_csv(csv_path)
    except (OSError, pd.errors.ParserError, UnicodeDecodeError) as exc:
        print(f"Skipping unreadable file {csv_path.name}: {exc}")
        continue
    if TARGET in candidate.columns:
        cars = candidate
        print(f"Loaded: {csv_path}")
        break
if cars is None:
    raise ValueError(
        f"No CSV in {DATA_DIR} has the required {TARGET!r} column. "
        "Check data/README.md for the expected dataset version."
    )

print(f"Rows and columns: {cars.shape}")
display(cars.head())

## 3. Inspect and clean the data

First inspect column types and missing values. Duplicate rows are removed, and rows without a usable selling price cannot be used as training examples. Feature values are kept in their original columns; the model pipeline will fill missing feature values appropriately without learning from the test set.

In [ ]:
print("Column types and non-null counts:")
display(cars.info())
print("Missing values by column:")
display(cars.isna().sum().sort_values(ascending=False))
print(f"Duplicate rows before cleaning: {cars.duplicated().sum()}")

clean_cars = cars.drop_duplicates().copy()
clean_cars[TARGET] = pd.to_numeric(clean_cars[TARGET], errors="coerce")
clean_cars = clean_cars.dropna(subset=[TARGET]).copy()
if len(clean_cars) < 2:
    raise ValueError("At least two rows with a numeric Selling_Price are required.")
print(f"Rows after removing duplicates and invalid target values: {len(clean_cars)}")
display(clean_cars.describe(include="all").transpose())

## 4. Explore the data with charts

These plots help us understand the target-price distribution and how it varies with vehicle age and kilometres driven. The plots describe this dataset; they do not prove that any feature causes a price change.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(data=clean_cars, x=TARGET, kde=True, ax=axes[0])
axes[0].set_title("Selling price distribution")
axes[0].set_xlabel("Selling price (dataset units)")

if "Year" in clean_cars.columns:
    sns.scatterplot(data=clean_cars, x="Year", y=TARGET, alpha=0.65, ax=axes[1])
    axes[1].set_title("Selling price by car year")
elif "Kms_Driven" in clean_cars.columns:
    sns.scatterplot(data=clean_cars, x="Kms_Driven", y=TARGET, alpha=0.65, ax=axes[1])
    axes[1].set_title("Selling price by kilometres driven")
else:
    axes[1].text(0.5, 0.5, "No Year or Kms_Driven column", ha="center", va="center")
    axes[1].set_axis_off()
plt.tight_layout()
plt.show()

categorical_columns = clean_cars.select_dtypes(exclude="number").columns.tolist()
if categorical_columns:
    chart_column = categorical_columns[0]
    top_values = clean_cars[chart_column].value_counts().head(10).index
    chart_data = clean_cars[clean_cars[chart_column].isin(top_values)]
    plt.figure(figsize=(10, 4))
    sns.boxplot(data=chart_data, x=chart_column, y=TARGET)
    plt.title(f"Selling price across common {chart_column} categories")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("No categorical columns are available for a category chart.")

## 5. Prepare features and train a model

We predict `Selling_Price` from the other columns. Numeric features get median-value imputation; text categories get most-common-value imputation and one-hot encoding. A Random Forest regressor is trained on 80% of the data. The remaining 20% is held out for evaluation. The fixed random seed makes the split repeatable.

In [ ]:
X = clean_cars.drop(columns=[TARGET])
y = clean_cars[TARGET]
numeric_features = X.select_dtypes(include="number").columns.tolist()
categorical_features = X.select_dtypes(exclude="number").columns.tolist()

transformers = []
if numeric_features:
    transformers.append(("numeric", SimpleImputer(strategy="median"), numeric_features))
if categorical_features:
    categorical_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("one_hot", OneHotEncoder(handle_unknown="ignore")),
    ])
    transformers.append(("categorical", categorical_pipeline, categorical_features))
if not transformers:
    raise ValueError("No usable feature columns remain after separating the target.")

preprocessor = ColumnTransformer(transformers=transformers, remainder="drop")
model = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=200, random_state=42, min_samples_leaf=2, n_jobs=-1
    )),
])
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
model.fit(X_train, y_train)
print(f"Training rows: {len(X_train)}; test rows: {len(X_test)}")

## 6. Evaluate on held-out data

- **MAE** is the average absolute prediction error, in the same price units as `Selling_Price`; lower is better.
- **RMSE** is also in price units and penalizes larger errors more heavily; lower is better.
- **R-squared** compares the model with a simple mean-price baseline; closer to 1 is generally better, but it can be negative when the model performs poorly.

These are results only for this real dataset and this held-out split. They are not a guarantee of future performance.

In [ ]:
predictions = model.predict(X_test)
mae = mean_absolute_error(y_test, predictions)
rmse = mean_squared_error(y_test, predictions) ** 0.5
r2 = r2_score(y_test, predictions)
print(f"Mean absolute error (MAE): {mae:.3f}")
print(f"Root mean squared error (RMSE): {rmse:.3f}")
print(f"R-squared (R2): {r2:.3f}")

plt.figure(figsize=(6, 5))
sns.scatterplot(x=y_test, y=predictions, alpha=0.7)
low = min(float(y_test.min()), float(predictions.min()))
high = max(float(y_test.max()), float(predictions.max()))
plt.plot([low, high], [low, high], linestyle="--", color="red", label="Perfect prediction")
plt.xlabel("Actual selling price")
plt.ylabel("Predicted selling price")
plt.title("Actual vs predicted prices on test data")
plt.legend()
plt.tight_layout()
plt.show()

## 7. Save the fitted model

After training and evaluation succeed, save the complete preprocessing-and-model pipeline. Keeping preprocessing inside the pipeline ensures new input receives the same transformations. Only load joblib model files from sources you trust.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
model_path = OUTPUT_DIR / "used_car_price_model.joblib"
joblib.dump(model, model_path)
print(f"Saved model: {model_path}")

## 8. Conclusions and next steps

This project demonstrates a full supervised-learning workflow: inspect and clean real data, explore it visually, preprocess numeric and categorical features, fit a regression model, and measure errors on held-out examples. Review the printed MAE and RMSE in the dataset's price units and the R-squared value before drawing conclusions. Improve the project by comparing additional models, checking unusual values, and validating performance with cross-validation. Do not claim a result unless this notebook has completed successfully with the real CarDekho dataset.

To create the shareable HTML report after all cells finish successfully, run this from the project folder:

```powershell
jupyter nbconvert --to html --execute notebooks/used_car_price_prediction.ipynb --output used_car_price_prediction.html --output-dir submission
```